In [1]:
# ============================================================
# 02.10 STEP 1 — PRODUCT / CATEGORY BASELINE
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. Load product master
# ------------------------------------------------------------

data_path = Path("../data/synthetic")

products_file = data_path / "products_master.csv"

products_df = pd.read_csv(products_file)

print("=" * 70)
print("02.10 STEP 1 — PRODUCT / CATEGORY BASELINE")
print("=" * 70)

print("\nProducts loaded:", len(products_df))
print("Columns:", list(products_df.columns))

# ------------------------------------------------------------
# 2. Category distribution
# ------------------------------------------------------------

category_summary = (
    products_df
    .groupby("category")
    .agg(
        product_count=("product_id", "nunique"),
        average_unit_price=("unit_price", "mean"),
        minimum_unit_price=("unit_price", "min"),
        maximum_unit_price=("unit_price", "max"),
        average_cost_price=("cost_price", "mean"),
        average_margin_pct=("profit_margin_pct", "mean")
    )
    .reset_index()
)

category_summary["product_percentage"] = (
    category_summary["product_count"]
    / len(products_df)
    * 100
)

print("\nCategory summary:")
print(
    category_summary
    .sort_values("product_count", ascending=False)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 3. Product status distribution
# ------------------------------------------------------------

print("\nProduct status distribution:")

status_summary = (
    products_df["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="product_count")
)

status_summary["percentage"] = (
    status_summary["product_count"]
    / len(products_df)
    * 100
)

print(
    status_summary.to_string(index=False)
)

# ------------------------------------------------------------
# 4. Launch-date range
# ------------------------------------------------------------

products_df["launch_date"] = pd.to_datetime(
    products_df["launch_date"]
)

print("\nProduct launch-date range:")
print(
    "Earliest:",
    products_df["launch_date"].min().date()
)

print(
    "Latest:",
    products_df["launch_date"].max().date()
)

# ------------------------------------------------------------
# 5. Basic validation
# ------------------------------------------------------------

print("\nBasic validation:")

print(
    "Unique product IDs:",
    products_df["product_id"].nunique()
)

print(
    "Duplicate product IDs:",
    products_df["product_id"].duplicated().sum()
)

print(
    "Missing categories:",
    products_df["category"].isna().sum()
)

print(
    "Missing unit prices:",
    products_df["unit_price"].isna().sum()
)

print(
    "Missing cost prices:",
    products_df["cost_price"].isna().sum()
)

print(
    "Missing launch dates:",
    products_df["launch_date"].isna().sum()
)

print("\n02.10 Step 1 baseline generated.")


02.10 STEP 1 — PRODUCT / CATEGORY BASELINE

Products loaded: 2500
Columns: ['product_id', 'product_name', 'category', 'subcategory', 'brand', 'supplier_name', 'unit_price', 'cost_price', 'profit_margin_pct', 'gst_percent', 'stock_quantity', 'launch_date', 'status', 'weight_kg', 'rating', 'warranty_months']

Category summary:
              category  product_count  average_unit_price  minimum_unit_price  maximum_unit_price  average_cost_price  average_margin_pct  product_percentage
               Fashion            509        30959.845560              404.82            68547.98        24834.135442           24.871179               20.36
Beauty & Personal Care            505        31643.905010              154.81            66237.44        25409.441545           24.805663               20.20
        Home & Kitchen            504        28559.002560              133.75            67695.24        22832.841071           25.108889               20.16
           Electronics            500    

In [2]:
# ============================================================
# 02.10 STEP 2 — PRODUCT MASTER GAP ANALYSIS
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. Load current product master
# ------------------------------------------------------------

data_path = Path("../data/synthetic")

products_file = data_path / "products_master.csv"

products_df = pd.read_csv(products_file)

# ------------------------------------------------------------
# 2. Frozen specification
# ------------------------------------------------------------

required_category_counts = {
    "Electronics": 230,
    "Home & Kitchen": 260,
    "Fashion": 300,
    "Beauty & Personal Care": 200,
    "Sports & Fitness": 210,
    "Grocery": 300
}

required_total_products = 1500

# ------------------------------------------------------------
# 3. Current category counts
# ------------------------------------------------------------

current_counts = (
    products_df["category"]
    .value_counts()
    .to_dict()
)

# ------------------------------------------------------------
# 4. Build comparison
# ------------------------------------------------------------

comparison = []

for category, required_count in required_category_counts.items():

    current_count = current_counts.get(
        category,
        0
    )

    comparison.append({
        "category": category,
        "required_products": required_count,
        "current_products": current_count,
        "difference": current_count - required_count
    })

comparison_df = pd.DataFrame(comparison)

# ------------------------------------------------------------
# 5. Display comparison
# ------------------------------------------------------------

print("=" * 70)
print("02.10 STEP 2 — PRODUCT MASTER GAP ANALYSIS")
print("=" * 70)

print("\nFrozen specification:")
print("Required total products:", required_total_products)

print("\nCurrent total products:")
print(len(products_df))

print("\nCategory comparison:")
print(
    comparison_df.to_string(index=False)
)

# ------------------------------------------------------------
# 6. Check missing approved categories
# ------------------------------------------------------------

approved_categories = set(
    required_category_counts.keys()
)

current_categories = set(
    products_df["category"].dropna().unique()
)

missing_categories = (
    approved_categories
    - current_categories
)

unexpected_categories = (
    current_categories
    - approved_categories
)

print("\nMissing approved categories:")
print(
    sorted(missing_categories)
)

print("\nUnexpected categories:")
print(
    sorted(unexpected_categories)
)

# ------------------------------------------------------------
# 7. Product ID validation
# ------------------------------------------------------------

print("\nProduct ID validation:")
print(
    "Unique product IDs:",
    products_df["product_id"].nunique()
)

print(
    "Duplicate product IDs:",
    products_df["product_id"].duplicated().sum()
)

# ------------------------------------------------------------
# 8. Final gap summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 2 SUMMARY")
print("=" * 70)

if (
    len(products_df) == required_total_products
    and not missing_categories
    and comparison_df["difference"].eq(0).all()
):
    print("Product master matches frozen specification.")
else:
    print("Product master requires correction before 02.10 behavior implementation.")

print("\n02.10 Step 2 analysis completed.")


02.10 STEP 2 — PRODUCT MASTER GAP ANALYSIS

Frozen specification:
Required total products: 1500

Current total products:
2500

Category comparison:
              category  required_products  current_products  difference
           Electronics                230               500         270
        Home & Kitchen                260               504         244
               Fashion                300               509         209
Beauty & Personal Care                200               505         305
      Sports & Fitness                210                 0        -210
               Grocery                300               482         182

Missing approved categories:
['Sports & Fitness']

Unexpected categories:
[]

Product ID validation:
Unique product IDs: 2500
Duplicate product IDs: 0

STEP 2 SUMMARY
Product master requires correction before 02.10 behavior implementation.

02.10 Step 2 analysis completed.


In [3]:
# ============================================================
# 02.10 STEP 3 — CURRENT PRODUCT USAGE & SALES MIX
# READ-ONLY VALIDATION
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

data_path = Path("../data/synthetic")

products_file = data_path / "products_master.csv"
sales_folder = data_path / "customer_behavior_adjusted"

# ------------------------------------------------------------
# 2. Load product master
# ------------------------------------------------------------

products_df = pd.read_csv(products_file)

product_lookup = products_df[
    ["product_id", "category", "unit_price", "cost_price"]
].drop_duplicates("product_id")

print("=" * 70)
print("02.10 STEP 3 — CURRENT PRODUCT USAGE & SALES MIX")
print("=" * 70)

print("\nProducts in master:", len(products_df))

# ------------------------------------------------------------
# 3. Read adjusted sales files
# ------------------------------------------------------------

sales_files = sorted(
    sales_folder.glob("sales_*.csv")
)

print("Sales files found:", len(sales_files))

sales_summary = []

for file_path in sales_files:

    sales_df = pd.read_csv(file_path)

    sales_summary.append(
        sales_df[
            [
                "sales_line_id",
                "product_id",
                "quantity",
                "unit_selling_price",
                "discount_pct",
                "order_status"
            ]
        ]
    )

sales = pd.concat(
    sales_summary,
    ignore_index=True
)

print("Sales rows loaded:", len(sales))

# ------------------------------------------------------------
# 4. Product reference validation
# ------------------------------------------------------------

sales_product_ids = set(
    sales["product_id"]
    .dropna()
    .unique()
)

master_product_ids = set(
    products_df["product_id"]
    .dropna()
    .unique()
)

missing_product_ids = (
    sales_product_ids
    - master_product_ids
)

print("\nProduct reference validation:")
print(
    "Unique product IDs used in sales:",
    len(sales_product_ids)
)

print(
    "Missing product IDs in master:",
    len(missing_product_ids)
)

if len(missing_product_ids) == 0:
    print("Sales → Product relationship: PASSED")
else:
    print("Sales → Product relationship: FAILED")

# ------------------------------------------------------------
# 5. Add product category
# ------------------------------------------------------------

sales = sales.merge(
    product_lookup[
        ["product_id", "category"]
    ],
    on="product_id",
    how="left"
)

# ------------------------------------------------------------
# 6. Calculate net sales value
# ------------------------------------------------------------

sales["net_sales_value"] = (
    sales["quantity"]
    * sales["unit_selling_price"]
    * (
        1
        - sales["discount_pct"] / 100
    )
)

# ------------------------------------------------------------
# 7. Category sales distribution
# ------------------------------------------------------------

category_sales = (
    sales
    .groupby("category")
    .agg(
        sales_lines=("sales_line_id", "count"),
        quantity=("quantity", "sum"),
        net_sales_value=("net_sales_value", "sum"),
        unique_products=("product_id", "nunique")
    )
    .reset_index()
)

category_sales["sales_line_percentage"] = (
    category_sales["sales_lines"]
    / len(sales)
    * 100
)

category_sales["revenue_percentage"] = (
    category_sales["net_sales_value"]
    / category_sales["net_sales_value"].sum()
    * 100
)

print("\nCurrent sales category distribution:")

print(
    category_sales
    .sort_values(
        "sales_line_percentage",
        ascending=False
    )
    .to_string(index=False)
)

# ------------------------------------------------------------
# 8. Product-level revenue concentration
# ------------------------------------------------------------

product_sales = (
    sales
    .groupby("product_id")
    .agg(
        sales_lines=("sales_line_id", "count"),
        quantity=("quantity", "sum"),
        net_sales_value=("net_sales_value", "sum")
    )
    .reset_index()
)

product_sales = product_sales.sort_values(
    "net_sales_value",
    ascending=False
).reset_index(drop=True)

product_sales["cumulative_revenue"] = (
    product_sales["net_sales_value"]
    .cumsum()
)

total_revenue = (
    product_sales["net_sales_value"].sum()
)

product_sales["cumulative_revenue_pct"] = (
    product_sales["cumulative_revenue"]
    / total_revenue
    * 100
)

# ------------------------------------------------------------
# 9. Top 20% product contribution
# ------------------------------------------------------------

top_20_count = max(
    1,
    int(len(product_sales) * 0.20)
)

top_20_revenue = (
    product_sales
    .head(top_20_count)["net_sales_value"]
    .sum()
)

top_20_revenue_pct = (
    top_20_revenue
    / total_revenue
    * 100
)

print("\nProduct concentration:")
print(
    "Unique products used:",
    len(product_sales)
)

print(
    "Top 20% product count:",
    top_20_count
)

print(
    "Top 20% revenue contribution:",
    round(
        top_20_revenue_pct,
        2
    ),
    "%"
)

# ------------------------------------------------------------
# 10. Final Step 3 summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("02.10 STEP 3 SUMMARY")
print("=" * 70)

print(
    "Sales rows analysed:",
    len(sales)
)

print(
    "Product reference check:",
    "PASSED"
    if len(missing_product_ids) == 0
    else "FAILED"
)

print(
    "Products used in sales:",
    len(sales_product_ids)
)

print(
    "Products in master:",
    len(master_product_ids)
)

print(
    "Top 20% revenue contribution:",
    round(
        top_20_revenue_pct,
        2
    ),
    "%"
)

print("\n02.10 Step 3 analysis completed.")

02.10 STEP 3 — CURRENT PRODUCT USAGE & SALES MIX

Products in master: 2500
Sales files found: 32
Sales rows loaded: 2118200

Product reference validation:
Unique product IDs used in sales: 2475
Missing product IDs in master: 0
Sales → Product relationship: PASSED

Current sales category distribution:
              category  sales_lines  quantity  net_sales_value  unique_products  sales_line_percentage  revenue_percentage
           Electronics       440090    703711     2.185002e+10              496              20.776603           21.180168
Beauty & Personal Care       434738    694846     2.155382e+10              502              20.523935           20.893050
               Fashion       423858    678878     2.078914e+10              505              20.010292           20.151808
        Home & Kitchen       417598    667653     1.907872e+10              495              19.714758           18.493826
               Grocery       401916    642687     1.989094e+10              477    

In [4]:
# ============================================================
# 02.10 STEP 4 — PRODUCT / CATEGORY BEHAVIOR WEIGHTS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Load product master
# ------------------------------------------------------------

data_path = Path("../data/synthetic")

products_file = data_path / "products_master.csv"

products_df = pd.read_csv(products_file)

products_df["launch_date"] = pd.to_datetime(
    products_df["launch_date"]
)

# ------------------------------------------------------------
# 2. Approved category demand weights
# ------------------------------------------------------------

category_demand_weights = {
    "Grocery": 0.30,
    "Fashion": 0.20,
    "Home & Kitchen": 0.15,
    "Beauty & Personal Care": 0.15,
    "Electronics": 0.10,
    "Sports & Fitness": 0.10
}

# ------------------------------------------------------------
# 3. Assign category demand weight
# ------------------------------------------------------------

products_df["category_demand_weight"] = (
    products_df["category"]
    .map(category_demand_weights)
)

# ------------------------------------------------------------
# 4. Product popularity rank within category
# ------------------------------------------------------------

products_df["product_rank_in_category"] = (
    products_df
    .groupby("category")["product_id"]
    .rank(
        method="first",
        ascending=True
    )
)

products_df["category_product_count"] = (
    products_df
    .groupby("category")["product_id"]
    .transform("count")
)

# ------------------------------------------------------------
# 5. Zipf-like product popularity weight
# ------------------------------------------------------------

products_df["product_popularity_weight"] = (
    1
    /
    np.power(
        products_df["product_rank_in_category"],
        1.15
    )
)

# ------------------------------------------------------------
# 6. Normalize popularity within category
# ------------------------------------------------------------

products_df["product_popularity_weight"] = (
    products_df
    .groupby("category")[
        "product_popularity_weight"
    ]
    .transform(
        lambda x: x / x.sum()
    )
)

# ------------------------------------------------------------
# 7. Lifecycle behavior
# ------------------------------------------------------------

reference_date = pd.Timestamp("2026-08-31")

product_age_days = (
    reference_date
    - products_df["launch_date"]
).dt.days.clip(lower=0)

# New products get ramp-up weight
products_df["lifecycle_weight"] = np.where(
    product_age_days <= 90,
    0.50,
    np.where(
        product_age_days <= 180,
        0.75,
        np.where(
            products_df["status"].eq("Discontinued"),
            0.25,
            1.00
        )
    )
)

# ------------------------------------------------------------
# 8. Combined product behavior weight
# ------------------------------------------------------------

products_df["behavior_weight"] = (
    products_df["category_demand_weight"].fillna(0)
    *
    products_df["product_popularity_weight"].fillna(0)
    *
    products_df["lifecycle_weight"]
)

# ------------------------------------------------------------
# 9. Validation
# ------------------------------------------------------------

print("=" * 70)
print("02.10 STEP 4 — PRODUCT / CATEGORY BEHAVIOR WEIGHTS")
print("=" * 70)

print("\nApproved category demand weights:")

for category, weight in category_demand_weights.items():
    print(
        f"{category}: {weight:.2f}"
    )

print("\nCurrent products with category weight:")
print(
    products_df[
        [
            "category",
            "product_id",
            "category_demand_weight",
            "product_popularity_weight",
            "lifecycle_weight",
            "behavior_weight"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print("\nMissing category weights:")
print(
    products_df["category_demand_weight"]
    .isna()
    .sum()
)

print(
    "\nMissing product popularity weights:",
    products_df["product_popularity_weight"]
    .isna()
    .sum()
)

print(
    "Missing lifecycle weights:",
    products_df["lifecycle_weight"]
    .isna()
    .sum()
)

print(
    "\nProducts with Sports & Fitness:",
    (
        products_df["category"]
        == "Sports & Fitness"
    ).sum()
)

print("\n02.10 Step 4 behavior-weight analysis completed.")

02.10 STEP 4 — PRODUCT / CATEGORY BEHAVIOR WEIGHTS

Approved category demand weights:
Grocery: 0.30
Fashion: 0.20
Home & Kitchen: 0.15
Beauty & Personal Care: 0.15
Electronics: 0.10
Sports & Fitness: 0.10

Current products with category weight:
              category product_id  category_demand_weight  product_popularity_weight  lifecycle_weight  behavior_weight
           Electronics PROD000001                    0.10                   0.215960              1.00         0.021596
Beauty & Personal Care PROD000002                    0.15                   0.215778              1.00         0.032367
               Grocery PROD000003                    0.30                   0.216636              1.00         0.064991
Beauty & Personal Care PROD000004                    0.15                   0.097235              1.00         0.014585
               Grocery PROD000005                    0.30                   0.097622              1.00         0.029287
           Electronics PROD000006  

In [5]:
# ============================================================
# 02.10 STEP 5 — SAFE PRODUCT REMAPPING PLAN
# READ-ONLY
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

data_path = Path("../data/synthetic")

products_file = data_path / "products_master.csv"
sales_folder = data_path / "customer_behavior_adjusted"

# ------------------------------------------------------------
# 2. Frozen product counts
# ------------------------------------------------------------

target_counts = {
    "Electronics": 230,
    "Home & Kitchen": 260,
    "Fashion": 300,
    "Beauty & Personal Care": 200,
    "Sports & Fitness": 210,
    "Grocery": 300
}

# ------------------------------------------------------------
# 3. Load product master
# ------------------------------------------------------------

products_df = pd.read_csv(products_file)

# ------------------------------------------------------------
# 4. Find products actually used by sales
# ------------------------------------------------------------

sales_files = sorted(
    sales_folder.glob("sales_*.csv")
)

used_product_ids = set()

for file_path in sales_files:

    sales_df = pd.read_csv(
        file_path,
        usecols=["product_id"]
    )

    used_product_ids.update(
        sales_df["product_id"]
        .dropna()
        .unique()
    )

# ------------------------------------------------------------
# 5. Mark product usage
# ------------------------------------------------------------

products_df["used_in_sales"] = (
    products_df["product_id"]
    .isin(used_product_ids)
)

# ------------------------------------------------------------
# 6. Current used-product counts
# ------------------------------------------------------------

used_counts = (
    products_df[
        products_df["used_in_sales"]
    ]
    .groupby("category")["product_id"]
    .nunique()
)

# ------------------------------------------------------------
# 7. Build remapping plan
# ------------------------------------------------------------

plan = []

for category, target_count in target_counts.items():

    current_used = int(
        used_counts.get(category, 0)
    )

    if current_used >= target_count:
        excess = (
            current_used
            - target_count
        )
        required_new = 0
    else:
        excess = 0
        required_new = (
            target_count
            - current_used
        )

    plan.append({
        "category": category,
        "target_products": target_count,
        "currently_used_products": current_used,
        "products_to_remap": excess,
        "new_products_required": required_new
    })

plan_df = pd.DataFrame(plan)

# ------------------------------------------------------------
# 8. Display plan
# ------------------------------------------------------------

print("=" * 75)
print("02.10 STEP 5 — SAFE PRODUCT REMAPPING PLAN")
print("=" * 75)

print("\nSales files analysed:", len(sales_files))
print(
    "Unique products currently used in sales:",
    len(used_product_ids)
)

print("\nProduct remapping plan:")

print(
    plan_df.to_string(index=False)
)

# ------------------------------------------------------------
# 9. Totals
# ------------------------------------------------------------

print("\nTotals:")

print(
    "Current used products:",
    len(used_product_ids)
)

print(
    "Target products:",
    sum(target_counts.values())
)

print(
    "Products requiring remapping:",
    plan_df["products_to_remap"].sum()
)

print(
    "New products required:",
    plan_df["new_products_required"].sum()
)

# ------------------------------------------------------------
# 10. Special Sports & Fitness check
# ------------------------------------------------------------

sports_current = int(
    used_counts.get(
        "Sports & Fitness",
        0
    )
)

sports_target = target_counts[
    "Sports & Fitness"
]

print("\nSports & Fitness:")

print(
    "Currently used:",
    sports_current
)

print(
    "Required:",
    sports_target
)

print(
    "New Sports & Fitness products required:",
    max(
        sports_target - sports_current,
        0
    )
)

# ------------------------------------------------------------
# 11. Final planning check
# ------------------------------------------------------------

if (
    plan_df["new_products_required"].sum()
    > 0
    or
    plan_df["products_to_remap"].sum()
    > 0
):

    print(
        "\nProduct master requires "
        "controlled remapping before "
        "02.10 behavior implementation."
    )

else:

    print(
        "\nNo product remapping required."
    )

print(
    "\n02.10 Step 5 planning completed."
)

02.10 STEP 5 — SAFE PRODUCT REMAPPING PLAN

Sales files analysed: 32
Unique products currently used in sales: 2475

Product remapping plan:
              category  target_products  currently_used_products  products_to_remap  new_products_required
           Electronics              230                      496                266                      0
        Home & Kitchen              260                      495                235                      0
               Fashion              300                      505                205                      0
Beauty & Personal Care              200                      502                302                      0
      Sports & Fitness              210                        0                  0                    210
               Grocery              300                      477                177                      0

Totals:
Current used products: 2475
Target products: 1500
Products requiring remapping: 1185
New products requ

In [6]:
# ============================================================
# 02.10 STEP 6 — CONTROLLED PRODUCT/CATEGORY IMPLEMENTATION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

data_path = Path("../data/synthetic")

products_file = data_path / "products_master.csv"
sales_folder = data_path / "customer_behavior_adjusted"

output_folder = data_path / "product_category_adjusted"
output_folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Approved product counts
# ------------------------------------------------------------

target_counts = {
    "Electronics": 230,
    "Home & Kitchen": 260,
    "Fashion": 300,
    "Beauty & Personal Care": 200,
    "Sports & Fitness": 210,
    "Grocery": 300
}

# ------------------------------------------------------------
# 3. Load product master
# ------------------------------------------------------------

products_df = pd.read_csv(products_file)

# ------------------------------------------------------------
# 4. Load sales product usage
# ------------------------------------------------------------

sales_files = sorted(
    sales_folder.glob("sales_*.csv")
)

used_product_ids = set()

for file_path in sales_files:

    sales_df = pd.read_csv(
        file_path,
        usecols=["product_id"]
    )

    used_product_ids.update(
        sales_df["product_id"]
        .dropna()
        .unique()
    )

# ------------------------------------------------------------
# 5. Identify currently used and unused products
# ------------------------------------------------------------

products_df["used_in_sales"] = (
    products_df["product_id"]
    .isin(used_product_ids)
)

used_products = (
    products_df[
        products_df["used_in_sales"]
    ]
    .copy()
)

unused_products = (
    products_df[
        ~products_df["used_in_sales"]
    ]
    .copy()
)

print("=" * 75)
print("02.10 STEP 6 — CONTROLLED PRODUCT/CATEGORY IMPLEMENTATION")
print("=" * 75)

print("\nCurrent master products:", len(products_df))
print("Products used in sales:", len(used_products))
print("Products unused in sales:", len(unused_products))

# ------------------------------------------------------------
# 6. Select products to retain
# ------------------------------------------------------------
# We retain the highest-value/popular products based on their
# current usage in sales. The remaining used products will be
# remapped to retained products.

sales_usage = {}

for file_path in sales_files:

    temp = pd.read_csv(
        file_path,
        usecols=["product_id"]
    )

    counts = (
        temp["product_id"]
        .value_counts()
    )

    for product_id, count in counts.items():

        if pd.notna(product_id):
            sales_usage[product_id] = (
                sales_usage.get(product_id, 0)
                + int(count)
            )

used_products["sales_line_count"] = (
    used_products["product_id"]
    .map(sales_usage)
    .fillna(0)
)

# ------------------------------------------------------------
# 7. Rank products within category
# ------------------------------------------------------------

used_products = used_products.sort_values(
    ["category", "sales_line_count"],
    ascending=[True, False]
)

# ------------------------------------------------------------
# 8. Select target products
# ------------------------------------------------------------

retained_parts = []

for category, target_count in target_counts.items():

    category_products = (
        used_products[
            used_products["category"] == category
        ]
        .copy()
    )

    # For Sports & Fitness, the current master has no products.
    # Existing unused products will be converted later.
    if len(category_products) > target_count:

        category_products = (
            category_products
            .head(target_count)
        )

    retained_parts.append(category_products)

retained_products = pd.concat(
    retained_parts,
    ignore_index=True
)

# ------------------------------------------------------------
# 9. Create Sports & Fitness products
# ------------------------------------------------------------

sports_needed = target_counts["Sports & Fitness"]

sports_existing = retained_products[
    retained_products["category"] == "Sports & Fitness"
]

sports_needed = (
    target_counts["Sports & Fitness"]
    - len(sports_existing)
)

print("\nSports & Fitness products required:", sports_needed)

if sports_needed > 0:

    available_for_sports = products_df[
        ~products_df["product_id"].isin(
            retained_products["product_id"]
        )
    ].copy()

    available_for_sports = (
        available_for_sports
        .sort_values(
            "product_id"
        )
        .head(sports_needed)
        .copy()
    )

    # Convert selected unused/retired records into
    # Sports & Fitness products.
    available_for_sports["category"] = (
        "Sports & Fitness"
    )

    retained_products = pd.concat(
        [
            retained_products,
            available_for_sports
        ],
        ignore_index=True
    )

# ------------------------------------------------------------
# 10. Final category count check
# ------------------------------------------------------------

final_category_counts = (
    retained_products
    .groupby("category")["product_id"]
    .nunique()
)

print("\nFinal planned product counts:")
print(final_category_counts)

# ------------------------------------------------------------
# 11. Final total
# ------------------------------------------------------------

final_product_count = (
    retained_products["product_id"]
    .nunique()
)

print(
    "\nFinal planned product count:",
    final_product_count
)

print(
    "Required product count:",
    sum(target_counts.values())
)

# ------------------------------------------------------------
# 12. Validation
# ------------------------------------------------------------

expected_total = sum(
    target_counts.values()
)

category_validation = all(
    int(final_category_counts.get(category, 0))
    == target_count
    for category, target_count
    in target_counts.items()
)

total_validation = (
    final_product_count
    == expected_total
)

print("\nCategory count validation:",
      "PASSED" if category_validation else "FAILED")

print("Total product validation:",
      "PASSED" if total_validation else "FAILED")

# ------------------------------------------------------------
# 13. Save planning output only
# ------------------------------------------------------------

planning_file = (
    output_folder /
    "product_category_control_plan.csv"
)

retained_products.to_csv(
    planning_file,
    index=False
)

print(
    "\nControl plan saved to:",
    planning_file
)

print(
    "\n02.10 Step 6 completed."
)

02.10 STEP 6 — CONTROLLED PRODUCT/CATEGORY IMPLEMENTATION

Current master products: 2500
Products used in sales: 2475
Products unused in sales: 25

Sports & Fitness products required: 210

Final planned product counts:
category
Beauty & Personal Care    200
Electronics               230
Fashion                   300
Grocery                   300
Home & Kitchen            260
Sports & Fitness          210
Name: product_id, dtype: int64

Final planned product count: 1500
Required product count: 1500

Category count validation: PASSED
Total product validation: PASSED

Control plan saved to: ..\data\synthetic\product_category_adjusted\product_category_control_plan.csv

02.10 Step 6 completed.


In [7]:
# ============================================================
# 02.10 STEP 7 — PRODUCT / SALES RELATIONSHIP VALIDATION
# READ-ONLY
# ============================================================

from pathlib import Path
import pandas as pd

data_path = Path("../data/synthetic")

products_file = data_path / "products_master.csv"
sales_folder = data_path / "customer_behavior_adjusted"
control_file = (
    data_path
    / "product_category_adjusted"
    / "product_category_control_plan.csv"
)

# ------------------------------------------------------------
# 1. Load control plan
# ------------------------------------------------------------

control_df = pd.read_csv(control_file)

# ------------------------------------------------------------
# 2. Basic product ID validation
# ------------------------------------------------------------

duplicate_product_ids = (
    control_df["product_id"]
    .duplicated()
    .sum()
)

missing_product_ids = (
    control_df["product_id"]
    .isna()
    .sum()
)

# ------------------------------------------------------------
# 3. Load product IDs referenced by sales
# ------------------------------------------------------------

sales_files = sorted(
    sales_folder.glob("sales_*.csv")
)

sales_product_ids = set()

for file_path in sales_files:

    sales_df = pd.read_csv(
        file_path,
        usecols=["product_id"]
    )

    sales_product_ids.update(
        sales_df["product_id"]
        .dropna()
        .unique()
    )

# ------------------------------------------------------------
# 4. Compare sales product IDs with control plan
# ------------------------------------------------------------

control_product_ids = set(
    control_df["product_id"]
)

sales_not_in_control = (
    sales_product_ids
    - control_product_ids
)

control_not_in_sales = (
    control_product_ids
    - sales_product_ids
)

# ------------------------------------------------------------
# 5. Category count validation
# ------------------------------------------------------------

category_counts = (
    control_df
    .groupby("category")["product_id"]
    .nunique()
    .sort_index()
)

# ------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------

print("=" * 75)
print("02.10 STEP 7 — PRODUCT / SALES RELATIONSHIP VALIDATION")
print("=" * 75)

print("\nControl-plan product count:")
print(control_df["product_id"].nunique())

print("\nDuplicate product IDs:")
print(duplicate_product_ids)

print("\nMissing product IDs:")
print(missing_product_ids)

print("\nUnique product IDs referenced by sales:")
print(len(sales_product_ids))

print("\nSales product IDs NOT present in control plan:")
print(len(sales_not_in_control))

print("\nControl-plan products NOT currently used by sales:")
print(len(control_not_in_sales))

print("\nCategory counts in control plan:")
print(category_counts)

# ------------------------------------------------------------
# 7. Final validation
# ------------------------------------------------------------

basic_validation = (
    control_df["product_id"].nunique() == 1500
    and duplicate_product_ids == 0
    and missing_product_ids == 0
)

relationship_validation = (
    len(sales_not_in_control) == 0
)

print("\nBasic control-plan validation:",
      "PASSED" if basic_validation else "FAILED")

print("Sales → product relationship:",
      "PASSED" if relationship_validation else "REVIEW REQUIRED")

if not relationship_validation:

    print(
        "\nIMPORTANT:"
        "\nThe control plan contains the planned 1,500 products,"
        "\nbut the existing sales files still reference products"
        "\nthat are outside this planned set."
        "\nNo files have been modified."
    )

print("\n02.10 Step 7 validation completed.")

02.10 STEP 7 — PRODUCT / SALES RELATIONSHIP VALIDATION

Control-plan product count:
1500

Duplicate product IDs:
0

Missing product IDs:
0

Unique product IDs referenced by sales:
2475

Sales product IDs NOT present in control plan:
979

Control-plan products NOT currently used by sales:
4

Category counts in control plan:
category
Beauty & Personal Care    200
Electronics               230
Fashion                   300
Grocery                   300
Home & Kitchen            260
Sports & Fitness          210
Name: product_id, dtype: int64

Basic control-plan validation: PASSED
Sales → product relationship: REVIEW REQUIRED

IMPORTANT:
The control plan contains the planned 1,500 products,
but the existing sales files still reference products
that are outside this planned set.
No files have been modified.

02.10 Step 7 validation completed.


In [1]:
# ============================================================
# 02.10 STEP 7 — CONTROLLED PRODUCT-ID REMAPPING
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

data_path = Path("../data/synthetic")

sales_input_folder = (
    data_path / "customer_behavior_adjusted"
)

control_file = (
    data_path
    / "product_category_adjusted"
    / "product_category_control_plan.csv"
)

sales_output_folder = (
    data_path
    / "product_category_adjusted"
    / "sales_remapped"
)

sales_output_folder.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Load product control plan
# ------------------------------------------------------------

control_df = pd.read_csv(control_file)

control_df["product_id"] = (
    control_df["product_id"]
    .astype(str)
)

control_product_ids = set(
    control_df["product_id"]
)

# ------------------------------------------------------------
# 3. Load original product master
# ------------------------------------------------------------

products_df = pd.read_csv(
    data_path / "products_master.csv"
)

products_df["product_id"] = (
    products_df["product_id"]
    .astype(str)
)

# ------------------------------------------------------------
# 4. Build source product → category lookup
# ------------------------------------------------------------

source_category_lookup = (
    products_df
    .drop_duplicates("product_id")
    .set_index("product_id")["category"]
    .to_dict()
)

# ------------------------------------------------------------
# 5. Build target products by category
# ------------------------------------------------------------

target_products_by_category = {}

for category in control_df["category"].unique():

    target_products_by_category[category] = (
        control_df[
            control_df["category"] == category
        ]["product_id"]
        .tolist()
    )

# ------------------------------------------------------------
# 6. Process sales files
# ------------------------------------------------------------

sales_files = sorted(
    sales_input_folder.glob("sales_*.csv")
)

print("=" * 75)
print("02.10 STEP 7 — CONTROLLED PRODUCT-ID REMAPPING")
print("=" * 75)

print(
    "\nSales files to process:",
    len(sales_files)
)

total_rows = 0
total_changed = 0
original_sales_line_ids = set()
adjusted_sales_line_ids = set()

rng = np.random.default_rng(20260922)

# ------------------------------------------------------------
# 7. Process each monthly file
# ------------------------------------------------------------

for file_path in sales_files:

    print("\nProcessing:", file_path.name)

    df = pd.read_csv(file_path)

    # Preserve original identifiers
    original_sales_line_ids.update(
        df["sales_line_id"].astype(str)
    )

    original_product_ids = (
        df["product_id"]
        .astype(str)
    )

    new_product_ids = original_product_ids.copy()

    # --------------------------------------------------------
    # Find product IDs outside the control plan
    # --------------------------------------------------------

    unmatched_mask = (
        ~original_product_ids.isin(
            control_product_ids
        )
    )

    unmatched_indices = (
        df.index[unmatched_mask]
        .tolist()
    )

    # --------------------------------------------------------
    # Remap unmatched products
    # --------------------------------------------------------

    for idx in unmatched_indices:

        old_product_id = str(
            df.at[idx, "product_id"]
        )

        source_category = (
            source_category_lookup.get(
                old_product_id
            )
        )

        # Prefer the same category
        target_pool = (
            target_products_by_category
            .get(source_category, [])
        )

        # If no same-category target exists,
        # use all approved target products.
        if not target_pool:

            target_pool = (
                control_df["product_id"]
                .tolist()
            )

        # Deterministic random selection
        new_product_id = rng.choice(
            target_pool
        )

        new_product_ids.loc[idx] = (
            new_product_id
        )

    # --------------------------------------------------------
    # Apply product ID changes
    # --------------------------------------------------------

    df["product_id"] = new_product_ids

    changed_count = (
        original_product_ids
        != new_product_ids
    ).sum()

    total_changed += int(changed_count)
    total_rows += len(df)

    # Preserve sales_line_id
    adjusted_sales_line_ids.update(
        df["sales_line_id"].astype(str)
    )

    # --------------------------------------------------------
    # Save adjusted file
    # --------------------------------------------------------

    output_file = (
        sales_output_folder
        / file_path.name
    )

    df.to_csv(
        output_file,
        index=False
    )

    print(
        "Rows:",
        len(df),
        "| Product IDs remapped:",
        int(changed_count),
        "| Saved:",
        output_file.name
    )

# ------------------------------------------------------------
# 8. Final relationship validation
# ------------------------------------------------------------

adjusted_product_ids = set()

adjusted_files = sorted(
    sales_output_folder.glob("sales_*.csv")
)

for file_path in adjusted_files:

    df = pd.read_csv(
        file_path,
        usecols=["product_id"]
    )

    adjusted_product_ids.update(
        df["product_id"]
        .astype(str)
        .unique()
    )

sales_outside_control = (
    adjusted_product_ids
    - control_product_ids
)

sales_line_id_preserved = (
    original_sales_line_ids
    == adjusted_sales_line_ids
)

# ------------------------------------------------------------
# 9. Final validation output
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("02.10 STEP 7 — REMAPPING VALIDATION")
print("=" * 75)

print(
    "\nOriginal total sales rows:",
    total_rows
)

print(
    "Total product IDs remapped:",
    total_changed
)

print(
    "Adjusted sales files:",
    len(adjusted_files)
)

print(
    "Unique product IDs after remapping:",
    len(adjusted_product_ids)
)

print(
    "Sales product IDs outside control plan:",
    len(sales_outside_control)
)

print(
    "Sales line ID preservation:",
    "PASSED"
    if sales_line_id_preserved
    else "FAILED"
)

print(
    "Product relationship:",
    "PASSED"
    if len(sales_outside_control) == 0
    else "FAILED"
)

# ------------------------------------------------------------
# 10. Final Step 7 status
# ------------------------------------------------------------

if (
    len(sales_outside_control) == 0
    and sales_line_id_preserved
):

    print(
        "\n02.10 STEP 7 VALIDATION: PASSED"
    )

else:

    print(
        "\n02.10 STEP 7 VALIDATION: REVIEW REQUIRED"
    )

02.10 STEP 7 — CONTROLLED PRODUCT-ID REMAPPING

Sales files to process: 32

Processing: sales_2024-01.csv
Rows: 53200 | Product IDs remapped: 2543 | Saved: sales_2024-01.csv

Processing: sales_2024-02.csv
Rows: 50400 | Product IDs remapped: 2495 | Saved: sales_2024-02.csv

Processing: sales_2024-03.csv
Rows: 56000 | Product IDs remapped: 3342 | Saved: sales_2024-03.csv

Processing: sales_2024-04.csv
Rows: 57100 | Product IDs remapped: 3807 | Saved: sales_2024-04.csv

Processing: sales_2024-05.csv
Rows: 58800 | Product IDs remapped: 4499 | Saved: sales_2024-05.csv

Processing: sales_2024-06.csv
Rows: 60500 | Product IDs remapped: 5319 | Saved: sales_2024-06.csv

Processing: sales_2024-07.csv
Rows: 57700 | Product IDs remapped: 5895 | Saved: sales_2024-07.csv

Processing: sales_2024-08.csv
Rows: 60500 | Product IDs remapped: 6880 | Saved: sales_2024-08.csv

Processing: sales_2024-09.csv
Rows: 64400 | Product IDs remapped: 7701 | Saved: sales_2024-09.csv

Processing: sales_2024-10.csv
Row

In [2]:
# ============================================================
# 02.10 STEP 8 — PRODUCT POPULARITY / PARETO WEIGHTS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

data_path = Path("../data/synthetic")

control_file = (
    data_path
    / "product_category_adjusted"
    / "product_category_control_plan.csv"
)

output_folder = (
    data_path
    / "product_category_adjusted"
)

# ------------------------------------------------------------
# 2. Load control plan
# ------------------------------------------------------------

products_df = pd.read_csv(control_file)

# ------------------------------------------------------------
# 3. Validate product IDs
# ------------------------------------------------------------

duplicate_ids = (
    products_df["product_id"]
    .duplicated()
    .sum()
)

missing_ids = (
    products_df["product_id"]
    .isna()
    .sum()
)

# ------------------------------------------------------------
# 4. Create deterministic Pareto-style popularity rank
# ------------------------------------------------------------

products_df = products_df.sort_values(
    ["category", "product_id"]
).reset_index(drop=True)

products_df["popularity_rank"] = (
    products_df
    .groupby("category")
    .cumcount()
    + 1
)

# ------------------------------------------------------------
# 5. Create Zipf-like popularity weight
# ------------------------------------------------------------

products_df["product_popularity_weight"] = (
    1
    /
    np.power(
        products_df["popularity_rank"],
        0.85
    )
)

# ------------------------------------------------------------
# 6. Normalize within category
# ------------------------------------------------------------

products_df["product_popularity_weight"] = (
    products_df
    .groupby("category")[
        "product_popularity_weight"
    ]
    .transform(
        lambda x: x / x.sum()
    )
)

# ------------------------------------------------------------
# 7. Category-level summary
# ------------------------------------------------------------

category_summary = (
    products_df
    .groupby("category")
    .agg(
        product_count=("product_id", "nunique"),
        total_weight=(
            "product_popularity_weight",
            "sum"
        ),
        max_weight=(
            "product_popularity_weight",
            "max"
        ),
        min_weight=(
            "product_popularity_weight",
            "min"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 8. Top-product concentration
# ------------------------------------------------------------

top20_rows = []

for category, group in products_df.groupby(
    "category"
):

    group = group.sort_values(
        "product_popularity_weight",
        ascending=False
    )

    top_n = max(
        1,
        int(len(group) * 0.20)
    )

    top20_weight = (
        group
        .head(top_n)[
            "product_popularity_weight"
        ]
        .sum()
    )

    top20_rows.append({
        "category": category,
        "product_count": len(group),
        "top_20_product_weight_pct":
            top20_weight * 100
    })

top20_df = pd.DataFrame(
    top20_rows
)

# ------------------------------------------------------------
# 9. Display results
# ------------------------------------------------------------

print("=" * 75)
print("02.10 STEP 8 — PRODUCT POPULARITY / PARETO WEIGHTS")
print("=" * 75)

print("\nProducts analysed:")
print(len(products_df))

print("\nDuplicate product IDs:")
print(duplicate_ids)

print("\nMissing product IDs:")
print(missing_ids)

print("\nCategory summary:")
print(
    category_summary.to_string(
        index=False
    )
)

print("\nTop 20% product weight by category:")
print(
    top20_df.to_string(
        index=False
    )
)

print("\nMost popular products:")

print(
    products_df[
        [
            "product_id",
            "category",
            "popularity_rank",
            "product_popularity_weight"
        ]
    ]
    .sort_values(
        "product_popularity_weight",
        ascending=False
    )
    .head(15)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 10. Validation
# ------------------------------------------------------------

weights_present = (
    products_df[
        "product_popularity_weight"
    ]
    .notna()
    .all()
)

weights_positive = (
    products_df[
        "product_popularity_weight"
    ]
    > 0
).all()

category_weight_sums = (
    products_df
    .groupby("category")[
        "product_popularity_weight"
    ]
    .sum()
)

category_weights_valid = (
    np.isclose(
        category_weight_sums.values,
        1.0
    ).all()
)

if (
    duplicate_ids == 0
    and missing_ids == 0
    and weights_present
    and weights_positive
    and category_weights_valid
):

    print(
        "\n02.10 Step 8 validation: PASSED"
    )

else:

    print(
        "\n02.10 Step 8 validation: FAILED"
    )

# ------------------------------------------------------------
# 11. Save controlled output
# ------------------------------------------------------------

output_file = (
    output_folder
    / "product_category_popularity_weights.csv"
)

products_df.to_csv(
    output_file,
    index=False
)

print(
    "\nPopularity-weighted product plan saved to:"
)

print(output_file)

print(
    "\n02.10 Step 8 completed."
)

02.10 STEP 8 — PRODUCT POPULARITY / PARETO WEIGHTS

Products analysed:
1500

Duplicate product IDs:
0

Missing product IDs:
0

Category summary:
              category  product_count  total_weight  max_weight  min_weight
Beauty & Personal Care            200           1.0    0.115417    0.001278
           Electronics            230           1.0    0.111404    0.001095
               Fashion            300           1.0    0.104295    0.000818
               Grocery            300           1.0    0.104295    0.000818
        Home & Kitchen            260           1.0    0.108043    0.000957
      Sports & Fitness            210           1.0    0.113993    0.001211

Top 20% product weight by category:
              category  product_count  top_20_product_weight_pct
Beauty & Personal Care            200                  63.649544
           Electronics            230                  64.146100
               Fashion            300                  65.033004
               Grocery    

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 65)
print("02.10 STEP 9 — FINAL PRODUCT BEHAVIOR WEIGHTS")
print("=" * 65)

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

data_path = Path.cwd().parent / "data" / "synthetic"

adjusted_path = data_path / "product_category_adjusted"

control_plan_path = (
    adjusted_path / "product_category_control_plan.csv"
)

popularity_path = (
    adjusted_path / "product_category_popularity_weights.csv"
)

output_path = (
    adjusted_path / "product_behavior_weights.csv"
)

# ------------------------------------------------------------
# 2. Load controlled product plan
# ------------------------------------------------------------

control_plan = pd.read_csv(control_plan_path)

print("\nControlled products loaded:", len(control_plan))

# ------------------------------------------------------------
# 3. Load Pareto popularity weights
# ------------------------------------------------------------

popularity = pd.read_csv(popularity_path)

print("Popularity-weighted products loaded:", len(popularity))

# ------------------------------------------------------------
# 4. Keep required columns
# ------------------------------------------------------------

required_control_columns = [
    "product_id",
    "category"
]

missing_control = [
    col for col in required_control_columns
    if col not in control_plan.columns
]

if missing_control:
    raise ValueError(
        f"Missing columns in control plan: {missing_control}"
    )

required_popularity_columns = [
    "product_id",
    "product_popularity_weight"
]

missing_popularity = [
    col for col in required_popularity_columns
    if col not in popularity.columns
]

if missing_popularity:
    raise ValueError(
        f"Missing columns in popularity weights: {missing_popularity}"
    )

control_plan = control_plan[
    required_control_columns
].drop_duplicates("product_id")

popularity = popularity[
    required_popularity_columns
].drop_duplicates("product_id")

# ------------------------------------------------------------
# 5. Merge popularity weights
# ------------------------------------------------------------

behavior = control_plan.merge(
    popularity,
    on="product_id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 6. Category demand weights
# ------------------------------------------------------------

category_weights = {
    "Grocery": 0.30,
    "Fashion": 0.20,
    "Home & Kitchen": 0.15,
    "Beauty & Personal Care": 0.15,
    "Electronics": 0.10,
    "Sports & Fitness": 0.10
}

behavior["category_demand_weight"] = (
    behavior["category"].map(category_weights)
)

# ------------------------------------------------------------
# 7. Lifecycle weight
# ------------------------------------------------------------

products_master = pd.read_csv(
    data_path / "products_master.csv"
)

lifecycle_columns = [
    "product_id",
    "launch_date",
    "status"
]

missing_lifecycle = [
    col for col in lifecycle_columns
    if col not in products_master.columns
]

if missing_lifecycle:
    raise ValueError(
        f"Missing lifecycle columns: {missing_lifecycle}"
    )

lifecycle = products_master[
    lifecycle_columns
].drop_duplicates("product_id")

behavior = behavior.merge(
    lifecycle,
    on="product_id",
    how="left",
    validate="one_to_one"
)

behavior["launch_date"] = pd.to_datetime(
    behavior["launch_date"],
    errors="coerce"
)

# ------------------------------------------------------------
# Lifecycle treatment
# ------------------------------------------------------------

behavior["lifecycle_weight"] = 1.00

behavior.loc[
    behavior["status"].astype(str).str.lower().eq("discontinued"),
    "lifecycle_weight"
] = 0.75

# Products launched after the simulation period
# receive zero availability weight.
simulation_end = pd.Timestamp("2026-08-31")

behavior.loc[
    behavior["launch_date"] > simulation_end,
    "lifecycle_weight"
] = 0.00

# ------------------------------------------------------------
# 8. Final behavior weight
# ------------------------------------------------------------

behavior["behavior_weight"] = (
    behavior["category_demand_weight"]
    * behavior["product_popularity_weight"]
    * behavior["lifecycle_weight"]
)

# ------------------------------------------------------------
# 9. Normalize weights within each category
# ------------------------------------------------------------

behavior["behavior_weight"] = (
    behavior
    .groupby("category")["behavior_weight"]
    .transform(
        lambda x: x / x.sum()
        if x.sum() > 0
        else x
    )
)

# ------------------------------------------------------------
# 10. Validation
# ------------------------------------------------------------

print("\nFinal product count:", len(behavior))

print(
    "Duplicate product IDs:",
    behavior["product_id"].duplicated().sum()
)

print(
    "Missing category weights:",
    behavior["category_demand_weight"].isna().sum()
)

print(
    "Missing popularity weights:",
    behavior["product_popularity_weight"].isna().sum()
)

print(
    "Missing lifecycle weights:",
    behavior["lifecycle_weight"].isna().sum()
)

print("\nFinal behavior weight by category:")

print(
    behavior.groupby("category")["behavior_weight"]
    .sum()
    .round(6)
)

# ------------------------------------------------------------
# 11. Save
# ------------------------------------------------------------

behavior.to_csv(
    output_path,
    index=False
)

print("\nFinal product behavior weights saved to:")
print(output_path)

print("\n02.10 Step 9 completed.")

02.10 STEP 9 — FINAL PRODUCT BEHAVIOR WEIGHTS

Controlled products loaded: 1500
Popularity-weighted products loaded: 1500

Final product count: 1500
Duplicate product IDs: 0
Missing category weights: 0
Missing popularity weights: 0
Missing lifecycle weights: 0

Final behavior weight by category:
category
Beauty & Personal Care    1.0
Electronics               1.0
Fashion                   1.0
Grocery                   1.0
Home & Kitchen            1.0
Sports & Fitness          1.0
Name: behavior_weight, dtype: float64

Final product behavior weights saved to:
c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\product_category_adjusted\product_behavior_weights.csv

02.10 Step 9 completed.


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 65)
print("02.10 STEP 10 — PRICE & MARGIN DISTRIBUTION VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

data_path = Path.cwd().parent / "data" / "synthetic"
adjusted_path = data_path / "product_category_adjusted"

control_plan_path = (
    adjusted_path / "product_category_control_plan.csv"
)

products_path = data_path / "products_master.csv"

# ------------------------------------------------------------
# 2. Load data
# ------------------------------------------------------------

control_plan = pd.read_csv(control_plan_path)

products = pd.read_csv(products_path)

print("\nControlled products:", len(control_plan))
print("Products in master:", len(products))

# ------------------------------------------------------------
# 3. Keep controlled 1,500 products
# ------------------------------------------------------------

controlled_ids = control_plan["product_id"].drop_duplicates()

products = products[
    products["product_id"].isin(controlled_ids)
].copy()

print(
    "Controlled products available in master:",
    len(products)
)

# ------------------------------------------------------------
# 4. Convert numeric fields
# ------------------------------------------------------------

products["unit_price"] = pd.to_numeric(
    products["unit_price"],
    errors="coerce"
)

products["cost_price"] = pd.to_numeric(
    products["cost_price"],
    errors="coerce"
)

products["profit_margin_pct"] = pd.to_numeric(
    products["profit_margin_pct"],
    errors="coerce"
)

# ------------------------------------------------------------
# 5. Basic validation
# ------------------------------------------------------------

print("\nMissing unit prices:",
      products["unit_price"].isna().sum())

print("Missing cost prices:",
      products["cost_price"].isna().sum())

print("Missing margins:",
      products["profit_margin_pct"].isna().sum())

print(
    "Non-positive unit prices:",
    (products["unit_price"] <= 0).sum()
)

print(
    "Non-positive cost prices:",
    (products["cost_price"] <= 0).sum()
)

print(
    "Cost >= selling price:",
    (products["cost_price"] >= products["unit_price"]).sum()
)

# ------------------------------------------------------------
# 6. Category price and margin summary
# ------------------------------------------------------------

category_summary = (
    products
    .groupby("category")
    .agg(
        product_count=("product_id", "nunique"),
        min_price=("unit_price", "min"),
        avg_price=("unit_price", "mean"),
        max_price=("unit_price", "max"),
        min_cost=("cost_price", "min"),
        avg_cost=("cost_price", "mean"),
        max_cost=("cost_price", "max"),
        min_margin_pct=("profit_margin_pct", "min"),
        avg_margin_pct=("profit_margin_pct", "mean"),
        max_margin_pct=("profit_margin_pct", "max")
    )
    .round(2)
)

print("\nCategory price and margin summary:")
print(category_summary)

# ------------------------------------------------------------
# 7. Overall price distribution
# ------------------------------------------------------------

price_percentiles = (
    products["unit_price"]
    .quantile([0, 0.25, 0.50, 0.75, 0.90, 0.95, 1.00])
    .round(2)
)

print("\nSelling-price percentiles:")
print(price_percentiles)

# ------------------------------------------------------------
# 8. Overall margin distribution
# ------------------------------------------------------------

margin_percentiles = (
    products["profit_margin_pct"]
    .quantile([0, 0.25, 0.50, 0.75, 0.90, 0.95, 1.00])
    .round(2)
)

print("\nProfit-margin percentiles:")
print(margin_percentiles)

# ------------------------------------------------------------
# 9. Price bands
# ------------------------------------------------------------

products["price_band"] = pd.cut(
    products["unit_price"],
    bins=[0, 5000, 15000, 30000, 50000, np.inf],
    labels=[
        "0-5K",
        "5K-15K",
        "15K-30K",
        "30K-50K",
        "50K+"
    ],
    include_lowest=True
)

print("\nPrice-band distribution:")
print(
    products["price_band"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 10. Margin bands
# ------------------------------------------------------------

products["margin_band"] = pd.cut(
    products["profit_margin_pct"],
    bins=[-np.inf, 10, 20, 30, 40, np.inf],
    labels=[
        "<=10%",
        "10-20%",
        "20-30%",
        "30-40%",
        "40%+"
    ]
)

print("\nMargin-band distribution:")
print(
    products["margin_band"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 11. Final validation
# ------------------------------------------------------------

validation_passed = (
    len(products) == 1500
    and products["product_id"].is_unique
    and products["unit_price"].notna().all()
    and products["cost_price"].notna().all()
    and products["profit_margin_pct"].notna().all()
    and (products["unit_price"] > 0).all()
    and (products["cost_price"] > 0).all()
)

if validation_passed:
    print("\n02.10 Step 10 validation: PASSED")
else:
    print("\n02.10 Step 10 validation: REVIEW REQUIRED")

02.10 STEP 10 — PRICE & MARGIN DISTRIBUTION VALIDATION

Controlled products: 1500
Products in master: 2500
Controlled products available in master: 1500

Missing unit prices: 0
Missing cost prices: 0
Missing margins: 0
Non-positive unit prices: 0
Non-positive cost prices: 0
Cost >= selling price: 0

Category price and margin summary:
                        product_count  min_price  avg_price  max_price  \
category                                                                 
Beauty & Personal Care            257     294.47   30141.22   65191.43   
Electronics                       277     560.07   30933.21   66765.77   
Fashion                           333     404.82   31163.97   68547.98   
Grocery                           338     187.21   31338.85   68466.35   
Home & Kitchen                    295     515.76   29132.40   67695.24   

                        min_cost  avg_cost  max_cost  min_margin_pct  \
category                                                               
B

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 65)
print("02.10 STEP 11 — PRODUCT LIFECYCLE VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

data_path = Path.cwd().parent / "data" / "synthetic"

adjusted_path = (
    data_path / "product_category_adjusted"
)

control_plan_path = (
    adjusted_path / "product_category_control_plan.csv"
)

products_path = (
    data_path / "products_master.csv"
)

# ------------------------------------------------------------
# 2. Load data
# ------------------------------------------------------------

control_plan = pd.read_csv(
    control_plan_path
)

products = pd.read_csv(
    products_path
)

# ------------------------------------------------------------
# 3. Keep controlled products
# ------------------------------------------------------------

controlled_ids = (
    control_plan["product_id"]
    .drop_duplicates()
)

products = products[
    products["product_id"].isin(controlled_ids)
].copy()

print("\nControlled products:", len(products))

# ------------------------------------------------------------
# 4. Prepare lifecycle fields
# ------------------------------------------------------------

products["launch_date"] = pd.to_datetime(
    products["launch_date"],
    errors="coerce"
)

products["status"] = (
    products["status"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# 5. Lifecycle status distribution
# ------------------------------------------------------------

print("\nProduct lifecycle status distribution:")

print(
    products["status"]
    .value_counts(dropna=False)
)

# ------------------------------------------------------------
# 6. Launch-date validation
# ------------------------------------------------------------

print("\nMissing launch dates:",
      products["launch_date"].isna().sum())

print(
    "Products launched after simulation end:",
    (
        products["launch_date"]
        > pd.Timestamp("2026-08-31")
    ).sum()
)

print(
    "Products launched before simulation start:",
    (
        products["launch_date"]
        < pd.Timestamp("2024-01-01")
    ).sum()
)

# ------------------------------------------------------------
# 7. Lifecycle weights
# ------------------------------------------------------------

products["lifecycle_weight"] = 1.00

# Discontinued products receive lower lifecycle weight
products.loc[
    products["status"].str.lower().eq("discontinued"),
    "lifecycle_weight"
] = 0.75

# Products launched after the historical simulation period
# are unavailable during the simulation
products.loc[
    products["launch_date"]
    > pd.Timestamp("2026-08-31"),
    "lifecycle_weight"
] = 0.00

# ------------------------------------------------------------
# 8. Launch-year distribution
# ------------------------------------------------------------

products["launch_year"] = (
    products["launch_date"]
    .dt.year
)

print("\nLaunch-year distribution:")

print(
    products["launch_year"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 9. Lifecycle summary
# ------------------------------------------------------------

lifecycle_summary = (
    products
    .groupby("status")
    .agg(
        product_count=("product_id", "nunique"),
        min_launch_date=("launch_date", "min"),
        max_launch_date=("launch_date", "max"),
        avg_lifecycle_weight=("lifecycle_weight", "mean")
    )
)

print("\nLifecycle summary:")
print(lifecycle_summary)

# ------------------------------------------------------------
# 10. Validate lifecycle logic
# ------------------------------------------------------------

duplicate_ids = (
    products["product_id"]
    .duplicated()
    .sum()
)

missing_launch_dates = (
    products["launch_date"]
    .isna()
    .sum()
)

invalid_weights = (
    (products["lifecycle_weight"] < 0)
    |
    (products["lifecycle_weight"] > 1)
).sum()

discontinued_count = (
    products["status"]
    .str.lower()
    .eq("discontinued")
    .sum()
)

active_count = (
    products["status"]
    .str.lower()
    .eq("active")
    .sum()
)

validation_passed = (
    len(products) == 1500
    and duplicate_ids == 0
    and missing_launch_dates == 0
    and invalid_weights == 0
    and discontinued_count > 0
    and active_count > 0
)

# ------------------------------------------------------------
# 11. Final result
# ------------------------------------------------------------

if validation_passed:
    print("\n02.10 Step 11 validation: PASSED")
else:
    print("\n02.10 Step 11 validation: REVIEW REQUIRED")

02.10 STEP 11 — PRODUCT LIFECYCLE VALIDATION

Controlled products: 1500

Product lifecycle status distribution:
status
Active          1344
Discontinued     156
Name: count, dtype: int64

Missing launch dates: 0
Products launched after simulation end: 4
Products launched before simulation start: 1102

Launch-year distribution:
launch_year
2021    135
2022    511
2023    456
2024    242
2025     99
2026     57
Name: count, dtype: int64

Lifecycle summary:
              product_count min_launch_date max_launch_date  \
status                                                        
Active                 1344      2021-09-15      2026-09-11   
Discontinued            156      2021-09-22      2026-08-18   

              avg_lifecycle_weight  
status                              
Active                    0.997024  
Discontinued              0.750000  

02.10 Step 11 validation: PASSED


In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 65)
print("02.10 STEP 12 — PRICE ELASTICITY ANALYSIS")
print("=" * 65)

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

data_path = Path.cwd().parent / "data" / "synthetic"

sales_files = sorted(
    data_path.glob("sales_*.csv")
)

products_path = (
    data_path / "products_master.csv"
)

# ------------------------------------------------------------
# 2. Load product master
# ------------------------------------------------------------

products = pd.read_csv(products_path)

products = products[
    [
        "product_id",
        "category",
        "unit_price",
        "cost_price"
    ]
].drop_duplicates("product_id")

# ------------------------------------------------------------
# 3. Load sales data
# ------------------------------------------------------------

sales_frames = []

for file in sales_files:
    df = pd.read_csv(file)

    required_columns = [
        "product_id",
        "quantity",
        "discount_pct"
    ]

    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{file.name} missing columns: {missing}"
        )

    sales_frames.append(
        df[required_columns]
    )

sales = pd.concat(
    sales_frames,
    ignore_index=True
)

print("\nSales files loaded:", len(sales_files))
print("Sales rows loaded:", len(sales))

# ------------------------------------------------------------
# 4. Merge category
# ------------------------------------------------------------

sales = sales.merge(
    products[
        ["product_id", "category"]
    ],
    on="product_id",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 5. Numeric conversion
# ------------------------------------------------------------

sales["quantity"] = pd.to_numeric(
    sales["quantity"],
    errors="coerce"
)

sales["discount_pct"] = pd.to_numeric(
    sales["discount_pct"],
    errors="coerce"
)

# ------------------------------------------------------------
# 6. Remove invalid analytical rows
# ------------------------------------------------------------

analysis = sales[
    sales["quantity"].notna()
    & sales["discount_pct"].notna()
    & sales["category"].notna()
].copy()

analysis = analysis[
    analysis["quantity"] > 0
]

# ------------------------------------------------------------
# 7. Discount groups
# ------------------------------------------------------------

analysis["discount_group"] = np.where(
    analysis["discount_pct"] > 0,
    "Discounted",
    "No Discount"
)

# ------------------------------------------------------------
# 8. Category-level analysis
# ------------------------------------------------------------

category_elasticity = (
    analysis
    .groupby(
        ["category", "discount_group"]
    )
    .agg(
        sales_lines=("quantity", "size"),
        total_quantity=("quantity", "sum"),
        avg_quantity=("quantity", "mean"),
        avg_discount_pct=("discount_pct", "mean")
    )
    .reset_index()
)

print("\nCategory / discount analysis:")
print(
    category_elasticity
    .to_string(index=False)
)

# ------------------------------------------------------------
# 9. Average quantity by discount state
# ------------------------------------------------------------

quantity_pivot = (
    analysis
    .pivot_table(
        index="category",
        columns="discount_group",
        values="quantity",
        aggfunc="mean"
    )
)

print("\nAverage quantity by category:")
print(
    quantity_pivot.round(4)
)

# ------------------------------------------------------------
# 10. Discount response ratio
# ------------------------------------------------------------

if (
    "Discounted" in quantity_pivot.columns
    and "No Discount" in quantity_pivot.columns
):

    quantity_pivot["discount_response_ratio"] = (
        quantity_pivot["Discounted"]
        /
        quantity_pivot["No Discount"]
    )

print("\nDiscount response ratio:")
print(
    quantity_pivot.round(4)
)

# ------------------------------------------------------------
# 11. Compare required categories
# ------------------------------------------------------------

required_categories = [
    "Fashion",
    "Beauty & Personal Care",
    "Grocery"
]

available_categories = [
    category
    for category in required_categories
    if category in quantity_pivot.index
]

print("\nRequired categories analysed:")
print(available_categories)

if "discount_response_ratio" in quantity_pivot.columns:

    print("\nPrice-response ratios:")

    for category in available_categories:
        print(
            category,
            "->",
            round(
                quantity_pivot.loc[
                    category,
                    "discount_response_ratio"
                ],
                4
            )
        )

# ------------------------------------------------------------
# 12. Basic data-quality validation
# ------------------------------------------------------------

missing_categories = (
    analysis["category"].isna().sum()
)

invalid_quantity = (
    analysis["quantity"] <= 0
).sum()

invalid_discount = (
    (analysis["discount_pct"] < 0)
    |
    (analysis["discount_pct"] > 100)
).sum()

print("\nValidation checks:")
print(
    "Missing categories:",
    missing_categories
)

print(
    "Invalid quantities:",
    invalid_quantity
)

print(
    "Invalid discount percentages:",
    invalid_discount
)

# ------------------------------------------------------------
# 13. Final result
# ------------------------------------------------------------

if (
    len(analysis) > 0
    and missing_categories == 0
    and invalid_quantity == 0
    and invalid_discount == 0
):
    print("\n02.10 Step 12 analysis: PASSED")
else:
    print("\n02.10 Step 12 analysis: REVIEW REQUIRED")

02.10 STEP 12 — PRICE ELASTICITY ANALYSIS

Sales files loaded: 32
Sales rows loaded: 2118200

Category / discount analysis:
              category discount_group  sales_lines  total_quantity  avg_quantity  avg_discount_pct
Beauty & Personal Care     Discounted        26571           42438      1.597155         17.462170
Beauty & Personal Care    No Discount       408167          652408      1.598385          0.000000
           Electronics     Discounted        32620           52146      1.598590         15.087199
           Electronics    No Discount       407470          651565      1.599050          0.000000
               Fashion     Discounted        27131           43487      1.602853         11.595531
               Fashion    No Discount       396727          635391      1.601582          0.000000
               Grocery     Discounted        40169           64264      1.599841         19.348215
               Grocery    No Discount       361747          578423      1.598971    

In [7]:
# ============================================================
# 02.10 STEP 13 — PRODUCT LIFECYCLE SALES BEHAVIOR
# ============================================================

from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data" / "synthetic"

PRODUCT_FILE = DATA_DIR / "products_master.csv"
SALES_DIR = DATA_DIR / "customer_behavior_adjusted"

# ------------------------------------------------------------
# 1. Load product master
# ------------------------------------------------------------

products = pd.read_csv(PRODUCT_FILE)

products["launch_date"] = pd.to_datetime(
    products["launch_date"],
    errors="coerce"
)

# ------------------------------------------------------------
# 2. Load all adjusted sales files
# ------------------------------------------------------------

sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

sales_list = []

for file in sales_files:
    df = pd.read_csv(file)

    df["order_date"] = pd.to_datetime(
        df["order_date"],
        errors="coerce"
    )

    sales_list.append(
        df[["sales_line_id", "product_id", "order_date"]]
    )

sales = pd.concat(
    sales_list,
    ignore_index=True
)

# ------------------------------------------------------------
# 3. Join sales with product launch/status information
# ------------------------------------------------------------

sales_check = sales.merge(
    products[
        [
            "product_id",
            "launch_date",
            "status"
        ]
    ],
    on="product_id",
    how="left"
)

# ------------------------------------------------------------
# 4. Validate product references
# ------------------------------------------------------------

missing_products = sales_check["launch_date"].isna().sum()

print("=" * 65)
print("02.10 STEP 13 — PRODUCT LIFECYCLE SALES BEHAVIOR")
print("=" * 65)

print(f"\nSales rows analysed: {len(sales_check)}")
print(f"Sales files analysed: {len(sales_files)}")
print(f"Missing product references: {missing_products}")

# ------------------------------------------------------------
# 5. Check sales before product launch
# ------------------------------------------------------------

pre_launch_sales = (
    sales_check["order_date"]
    < sales_check["launch_date"]
)

pre_launch_count = pre_launch_sales.sum()

print(f"Sales before product launch: {pre_launch_count}")

# ------------------------------------------------------------
# 6. Sales by product lifecycle status
# ------------------------------------------------------------

status_summary = (
    sales_check
    .groupby("status")
    .size()
    .reset_index(name="sales_lines")
)

print("\nSales by product status:")
print(status_summary.to_string(index=False))

# ------------------------------------------------------------
# 7. Final validation
# ------------------------------------------------------------

if missing_products == 0 and pre_launch_count == 0:
    print("\n02.10 Step 13 validation: PASSED")
else:
    print("\n02.10 Step 13 validation: REVIEW REQUIRED")

02.10 STEP 13 — PRODUCT LIFECYCLE SALES BEHAVIOR

Sales rows analysed: 2118200
Sales files analysed: 32
Missing product references: 0
Sales before product launch: 0

Sales by product status:
      status  sales_lines
      Active      1896151
Discontinued       222049

02.10 Step 13 validation: PASSED


In [8]:
# ============================================================
# 02.10 STEP 14 — BASKET AFFINITY ANALYSIS
# ============================================================

from pathlib import Path
from itertools import combinations
from collections import Counter
import pandas as pd

BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data" / "synthetic"

PRODUCT_FILE = DATA_DIR / "products_master.csv"
SALES_DIR = DATA_DIR / "customer_behavior_adjusted"

# ------------------------------------------------------------
# 1. Load product master
# ------------------------------------------------------------

products = pd.read_csv(PRODUCT_FILE)

product_category = products[
    ["product_id", "category"]
].drop_duplicates("product_id")

# ------------------------------------------------------------
# 2. Analyse category combinations within orders
# ------------------------------------------------------------

sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

pair_counter = Counter()
orders_with_multiple_categories = 0
total_orders = 0

for file in sales_files:

    sales = pd.read_csv(
        file,
        usecols=["order_id", "product_id"]
    )

    sales = sales.merge(
        product_category,
        on="product_id",
        how="left"
    )

    sales = sales.dropna(
        subset=["order_id", "category"]
    )

    order_categories = (
        sales
        .groupby("order_id")["category"]
        .unique()
    )

    total_orders += len(order_categories)

    for categories in order_categories:

        categories = sorted(set(categories))

        if len(categories) >= 2:
            orders_with_multiple_categories += 1

            for pair in combinations(categories, 2):
                pair_counter[pair] += 1

# ------------------------------------------------------------
# 3. Create affinity summary
# ------------------------------------------------------------

affinity_df = pd.DataFrame(
    [
        {
            "category_1": pair[0],
            "category_2": pair[1],
            "orders_with_pair": count
        }
        for pair, count in pair_counter.items()
    ]
)

if not affinity_df.empty:

    affinity_df["pair_percentage"] = (
        affinity_df["orders_with_pair"]
        / total_orders
        * 100
    )

    affinity_df = affinity_df.sort_values(
        "orders_with_pair",
        ascending=False
    )

# ------------------------------------------------------------
# 4. Display results
# ------------------------------------------------------------

print("=" * 65)
print("02.10 STEP 14 — BASKET AFFINITY ANALYSIS")
print("=" * 65)

print(f"\nSales files analysed: {len(sales_files)}")
print(f"Total orders analysed: {total_orders}")
print(
    f"Orders containing multiple categories: "
    f"{orders_with_multiple_categories}"
)

print("\nTop category combinations:")

if not affinity_df.empty:
    print(
        affinity_df.head(15).to_string(
            index=False
        )
    )
else:
    print("No multi-category combinations found.")

# ------------------------------------------------------------
# 5. Required affinity checks
# ------------------------------------------------------------

required_pairs = [
    ("Electronics", "Home & Kitchen"),
    ("Beauty & Personal Care", "Fashion"),
    ("Fashion", "Sports & Fitness")
]

found_pairs = set(
    zip(
        affinity_df["category_1"],
        affinity_df["category_2"]
    )
) if not affinity_df.empty else set()

print("\nRequired affinity combinations:")

for pair in required_pairs:

    pair_sorted = tuple(sorted(pair))

    if pair_sorted in found_pairs:
        print(f"{pair[0]} + {pair[1]}: FOUND")
    else:
        print(f"{pair[0]} + {pair[1]}: NOT FOUND")

# ------------------------------------------------------------
# 6. Final validation
# ------------------------------------------------------------

if (
    total_orders > 0
    and orders_with_multiple_categories > 0
    and not affinity_df.empty
):
    print("\n02.10 Step 14 validation: PASSED")
else:
    print("\n02.10 Step 14 validation: REVIEW REQUIRED")

02.10 STEP 14 — BASKET AFFINITY ANALYSIS

Sales files analysed: 32
Total orders analysed: 1707574
Orders containing multiple categories: 284073

Top category combinations:
            category_1     category_2  orders_with_pair  pair_percentage
Beauty & Personal Care    Electronics             37221         2.179759
           Electronics        Fashion             36360         2.129337
Beauty & Personal Care        Fashion             36056         2.111534
           Electronics Home & Kitchen             35994         2.107903
Beauty & Personal Care Home & Kitchen             35487         2.078212
           Electronics        Grocery             34543         2.022928
               Fashion Home & Kitchen             34333         2.010630
Beauty & Personal Care        Grocery             34305         2.008991
               Fashion        Grocery             33206         1.944630
               Grocery Home & Kitchen             32902         1.926827

Required affinity combin

In [11]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent
SALES_DIR = BASE_DIR / "data" / "synthetic"

sample_file = SALES_DIR / "sales_2024-01.csv"

sample_sales = pd.read_csv(sample_file, nrows=5)

print("=" * 70)
print("02.10 STEP 15 — SALES COLUMN CHECK")
print("=" * 70)

print("\nSales file:")
print(sample_file)

print("\nActual sales columns:")
for column in sample_sales.columns:
    print(column)

print("\nNumber of columns:", len(sample_sales.columns))

02.10 STEP 15 — SALES COLUMN CHECK

Sales file:
c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\sales_2024-01.csv

Actual sales columns:
sales_line_id
order_id
order_date
order_time
customer_id
product_id
store_id
salesperson_id
promotion_id
sales_channel
quantity
unit_selling_price
discount_pct
payment_method
order_status

Number of columns: 15


In [12]:
import pandas as pd

# Load sample sales data
sample_sales = pd.read_csv(sample_file)

# Calculate line-level gross and net sales values
sample_sales["gross_sales_value"] = (
    sample_sales["quantity"] *
    sample_sales["unit_selling_price"]
)

sample_sales["discount_value"] = (
    sample_sales["gross_sales_value"] *
    sample_sales["discount_pct"] / 100
)

sample_sales["net_sales_value"] = (
    sample_sales["gross_sales_value"] -
    sample_sales["discount_value"]
)

print("=" * 70)
print("02.10 STEP 15 — SALES VALUE CALCULATION CHECK")
print("=" * 70)

print("\nRows checked:", len(sample_sales))

print(
    "Negative net sales values:",
    (sample_sales["net_sales_value"] < 0).sum()
)

print(
    "Zero/negative quantities:",
    (sample_sales["quantity"] <= 0).sum()
)

print(
    "Invalid discount percentages:",
    (
        (sample_sales["discount_pct"] < 0) |
        (sample_sales["discount_pct"] > 100)
    ).sum()
)

print("\nSample calculated values:")
print(
    sample_sales[
        [
            "quantity",
            "unit_selling_price",
            "discount_pct",
            "gross_sales_value",
            "discount_value",
            "net_sales_value"
        ]
    ].head()
)

print("\n02.10 Step 15 sales-value calculation check completed.")

02.10 STEP 15 — SALES VALUE CALCULATION CHECK

Rows checked: 53200
Negative net sales values: 0
Zero/negative quantities: 0
Invalid discount percentages: 0

Sample calculated values:
   quantity  unit_selling_price  discount_pct  gross_sales_value  \
0         1            11583.24           0.0           11583.24   
1         3            55589.49           0.0          166768.47   
2         1            36375.45           0.0           36375.45   
3         1            22541.20           0.0           22541.20   
4         1            32466.25           0.0           32466.25   

   discount_value  net_sales_value  
0             0.0         11583.24  
1             0.0        166768.47  
2             0.0         36375.45  
3             0.0         22541.20  
4             0.0         32466.25  

02.10 Step 15 sales-value calculation check completed.


In [14]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent

PRODUCT_DIR = BASE_DIR / "data" / "synthetic" / "product_category_adjusted"
SALES_DIR = BASE_DIR / "data" / "synthetic"

CONTROL_FILE = PRODUCT_DIR / "product_category_control_plan.csv"
WEIGHT_FILE = PRODUCT_DIR / "product_behavior_weights.csv"

print("=" * 70)
print("02.10 STEP 16 — FINAL PRODUCT/CATEGORY BEHAVIOR VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load controlled product set
# ------------------------------------------------------------

control_df = pd.read_csv(CONTROL_FILE)

print("\nControlled products:", len(control_df))

print("\nControlled category distribution:")
print(
    control_df["category"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 2. Load final product behavior weights
# ------------------------------------------------------------

weights_df = pd.read_csv(WEIGHT_FILE)

print("\nProduct behavior weights:", len(weights_df))

# ------------------------------------------------------------
# 3. Validate product IDs
# ------------------------------------------------------------

duplicate_products = weights_df["product_id"].duplicated().sum()

missing_categories = weights_df["category"].isna().sum()

missing_behavior_weights = weights_df["behavior_weight"].isna().sum()

print("\nDuplicate product IDs:", duplicate_products)
print("Missing categories:", missing_categories)
print("Missing behavior weights:", missing_behavior_weights)

# ------------------------------------------------------------
# 4. Validate category coverage
# ------------------------------------------------------------

control_categories = set(control_df["category"].dropna().unique())
weight_categories = set(weights_df["category"].dropna().unique())

missing_categories_from_weights = control_categories - weight_categories
unexpected_categories = weight_categories - control_categories

print("\nCategories missing from behavior weights:")
print(missing_categories_from_weights)

print("\nUnexpected categories:")
print(unexpected_categories)

# ------------------------------------------------------------
# 5. Validate behavior weights
# ------------------------------------------------------------

invalid_weights = (
    weights_df["behavior_weight"].isna()
    | (weights_df["behavior_weight"] <= 0)
)

print(
    "\nInvalid behavior weights:",
    invalid_weights.sum()
)

# ------------------------------------------------------------
# 6. Category-level behavior summary
# ------------------------------------------------------------

category_summary = (
    weights_df
    .groupby("category")
    .agg(
        product_count=("product_id", "nunique"),
        total_behavior_weight=("behavior_weight", "sum"),
        avg_behavior_weight=("behavior_weight", "mean"),
        min_behavior_weight=("behavior_weight", "min"),
        max_behavior_weight=("behavior_weight", "max")
    )
    .reset_index()
)

print("\nCategory behavior summary:")
print(category_summary.to_string(index=False))

# ------------------------------------------------------------
# 7. Final validation
# ------------------------------------------------------------

validation_passed = (
    len(weights_df) == 1500
    and duplicate_products == 0
    and missing_categories == 0
    and missing_behavior_weights == 0
    and len(missing_categories_from_weights) == 0
    and len(unexpected_categories) == 0
    and invalid_weights.sum() == 0
)

if validation_passed:
    print("\n02.10 Step 16 validation: PASSED")
else:
    print("\n02.10 Step 16 validation: REVIEW REQUIRED")

02.10 STEP 16 — FINAL PRODUCT/CATEGORY BEHAVIOR VALIDATION

Controlled products: 1500

Controlled category distribution:
category
Beauty & Personal Care    200
Electronics               230
Fashion                   300
Grocery                   300
Home & Kitchen            260
Sports & Fitness          210
Name: count, dtype: int64

Product behavior weights: 1500

Duplicate product IDs: 0
Missing categories: 0
Missing behavior weights: 0

Categories missing from behavior weights:
set()

Unexpected categories:
set()

Invalid behavior weights: 4

Category behavior summary:
              category  product_count  total_behavior_weight  avg_behavior_weight  min_behavior_weight  max_behavior_weight
Beauty & Personal Care            200                    1.0             0.005000             0.001003             0.118734
           Electronics            230                    1.0             0.004348             0.000844             0.114026
               Fashion            300           

In [15]:
# Find products with zero or negative behavior weights

invalid_weight_products = weights_df[
    weights_df["behavior_weight"] <= 0
].copy()

print("=" * 70)
print("02.10 STEP 16 — INVALID PRODUCT BEHAVIOR WEIGHTS")
print("=" * 70)

print("\nProducts with zero/negative behavior weight:")
print(len(invalid_weight_products))

if len(invalid_weight_products) > 0:
    print("\nAffected products:")
    print(
        invalid_weight_products[
            ["product_id", "category", "behavior_weight"]
        ].to_string(index=False)
    )
else:
    print("\nNo invalid behavior weights found.")

02.10 STEP 16 — INVALID PRODUCT BEHAVIOR WEIGHTS

Products with zero/negative behavior weight:
4

Affected products:
product_id         category  behavior_weight
PROD000024 Sports & Fitness              0.0
PROD000069 Sports & Fitness              0.0
PROD000280 Sports & Fitness              0.0
PROD000302 Sports & Fitness              0.0


In [16]:
# ------------------------------------------------------------
# 02.10 STEP 16 — FIX ZERO PRODUCT BEHAVIOR WEIGHTS
# ------------------------------------------------------------

ZERO_WEIGHT_FLOOR = 0.000001

zero_mask = weights_df["behavior_weight"] <= 0

print("=" * 70)
print("02.10 STEP 16 — ZERO-WEIGHT CORRECTION")
print("=" * 70)

print("\nProducts requiring correction:", zero_mask.sum())

print(
    weights_df.loc[
        zero_mask,
        ["product_id", "category", "behavior_weight"]
    ].to_string(index=False)
)

# Apply a very small positive floor
weights_df.loc[zero_mask, "behavior_weight"] = ZERO_WEIGHT_FLOOR

# Renormalize weights within each category
weights_df["behavior_weight"] = (
    weights_df
    .groupby("category")["behavior_weight"]
    .transform(lambda x: x / x.sum())
)

print("\nZero-weight products after correction:")
print(
    (weights_df["behavior_weight"] <= 0).sum()
)

print("\nMinimum behavior weight:")
print(weights_df["behavior_weight"].min())

# Save corrected weights
weights_df.to_csv(WEIGHT_FILE, index=False)

print("\nCorrected product behavior weights saved to:")
print(WEIGHT_FILE)

print("\n02.10 Step 16 correction completed.")

02.10 STEP 16 — ZERO-WEIGHT CORRECTION

Products requiring correction: 4
product_id         category  behavior_weight
PROD000024 Sports & Fitness              0.0
PROD000069 Sports & Fitness              0.0
PROD000280 Sports & Fitness              0.0
PROD000302 Sports & Fitness              0.0

Zero-weight products after correction:
0

Minimum behavior weight:
9.999960000160107e-07

Corrected product behavior weights saved to:
c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\product_category_adjusted\product_behavior_weights.csv

02.10 Step 16 correction completed.


In [17]:
import pandas as pd
from pathlib import Path

print("=" * 70)
print("02.10 STEP 16 — FINAL PRODUCT/CATEGORY BEHAVIOR VALIDATION")
print("=" * 70)

BASE_DIR = Path("..")
WEIGHT_FILE = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "product_category_adjusted"
    / "product_behavior_weights.csv"
)

df = pd.read_csv(WEIGHT_FILE)

duplicate_product_ids = df["product_id"].duplicated().sum()
missing_categories = df["category"].isna().sum()
missing_behavior_weights = df["behavior_weight"].isna().sum()

expected_categories = {
    "Beauty & Personal Care",
    "Electronics",
    "Fashion",
    "Grocery",
    "Home & Kitchen",
    "Sports & Fitness"
}

actual_categories = set(df["category"].dropna().unique())
missing_categories_from_weights = expected_categories - actual_categories

zero_or_negative_weights = (df["behavior_weight"] <= 0).sum()

print(f"\nProducts validated: {len(df)}")
print(f"Duplicate product IDs: {duplicate_product_ids}")
print(f"Missing categories: {missing_categories}")
print(f"Missing behavior weights: {missing_behavior_weights}")
print(f"Categories missing from behavior weights: {missing_categories_from_weights}")
print(f"Zero/negative behavior weights: {zero_or_negative_weights}")

if (
    len(df) == 1500
    and duplicate_product_ids == 0
    and missing_categories == 0
    and missing_behavior_weights == 0
    and len(missing_categories_from_weights) == 0
    and zero_or_negative_weights == 0
):
    print("\n02.10 Step 16 validation: PASSED")
else:
    print("\n02.10 Step 16 validation: REVIEW REQUIRED")

02.10 STEP 16 — FINAL PRODUCT/CATEGORY BEHAVIOR VALIDATION

Products validated: 1500
Duplicate product IDs: 0
Missing categories: 0
Missing behavior weights: 0
Categories missing from behavior weights: set()
Zero/negative behavior weights: 0

02.10 Step 16 validation: PASSED
